# C08: MERT-95M versus MERT-330M

Frozen pretrained encoder changes from 768 to 1024 output features. The classifier remains a 64-channel three-layer residual Conv1D with GELU, max pooling, and a 64-to-32-to-2 MLP (GELU, dropout 0.1) with two-class softmax. Training uses class-balanced cross-entropy on raw logits; fake is class 1. AdamW weight decay is 0.01; other training settings are unchanged. Both backbones train fresh heads with seeds41–43 (six fits). MERT95 reuses verified cached features; prior unregularized checkpoints are retained only as historical inputs. MERT330 uses the same raw audio and separation, newly extracted once. Tune-only selection precedes historical evaluation. Original submission ZIP is preserved.

In [ ]:
from temporal_backbone import main
main()

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
out=Path('results')
selection=json.loads((out/'selection.json').read_text())
lookup={r['name']:r for r in json.loads((out/'test_report.json').read_text())['results']}
summary=[]
for group in selection['summaries']:
    rows=[lookup[k] for k in group['candidates']]
    summary.append(dict(name=group['name'],tune_eer=group['music_eer_mean'],tune_bce=group['balanced_bce_mean'],
      test_music_eer=float(np.mean([r['music_eer'] for r in rows])),
      test_file_eer=float(np.mean([r['metrics']['metrics']['file_eer'] for r in rows]))))
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
for ax,key,title in zip(axes,['tune_eer','test_music_eer','test_file_eer'],['Tune music EER','Historical music EER','Historical file EER']):
    bars=ax.bar(['MERT95 / 768','MERT330 / 1024'],[r[key] for r in summary],color=['#6c8eb4','#ee9b49'])
    ax.bar_label(bars,fmt='%.3f');ax.set_ylim(0,.65);ax.set_title(title+' (lower better)');ax.grid(axis='y',alpha=.2)
fig.savefig(out/'comparison.png',dpi=170);plt.show()
(out/'summary.json').write_text(json.dumps(dict(groups=summary,winner=selection['winner']['name']),indent=2))
print(json.dumps(summary,indent=2))